In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Create directory structure
!mkdir -p "/content/drive/MyDrive/GehlerShiBenchmark/dataset/images"
%cd "/content/drive/MyDrive/GehlerShiBenchmark/dataset"

/content/drive/MyDrive/GehlerShiBenchmark/dataset


In [ ]:
# 3. Download the Ground Truth Illuminants
!wget https://www.cs.sfu.ca/~colour/data/shi_gehler/real_illum_568.mat

--2026-10-11 21:30:07--  https://www.cs.sfu.ca/~colour/data/shi_gehler/real_illum_568.mat
Resolving www.cs.sfu.ca (www.cs.sfu.ca)... 142.58.226.71
Connecting to www.cs.sfu.ca (www.cs.sfu.ca)|142.58.226.71|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-10-11 21:30:08 ERROR 404: Not Found.



In [ ]:
# 1. Authenticate Kaggle
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# 2. Download the full Gehler-Shi mirror
!kaggle datasets download -d astritmt/gehler-shi-dataset

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


In [ ]:
# Clean up any previous empty file and download directly from a verified public repository using curl with location-redirects enabled
!rm -f real_illum_568.mat
!curl -L -o real_illum_568.mat "https://github.com/creed92/Color-Constancy/raw/master/real_illum_568.mat"
!ls -la real_illum_568.mat

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  263k    0  263k    0     0  1310k      0 --:--:-- --:--:-- --:--:-- 1321k
-rw------- 1 root root 269381 Oct 11 21:47 real_illum_568.mat


In [ ]:
# Επιστροφή στον αρχικό κατάλογο εργασίας
%cd /content

# Καθαρισμός του άδειου ή ελαττωματικού αρχείου
!rm -f 1D_png.zip

# Εγκατάσταση/Ενημέρωση του gdown για αποφυγή προβλημάτων με μεγάλα αρχεία
!pip install --upgrade -q gdown

# Λήψη του 1D_png.zip χρησιμοποιώντας το επιβεβαιωμένο Google Drive File ID από το Kernel State
print("Downloading 1D_png.zip via gdown...")
!gdown --id 1E60vskD8YI8n0j3_D6O0i0eZ5P5bAksf --output 1D_png.zip

# Επιβεβαίωση λήψης και ελέγχου του αρχείου zip
print("\nVerifying downloaded file:")
!ls -lh 1D_png.zip
!file 1D_png.zip

/content
usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--cookies FILE] [--cookies-from-browser BROWSER] [--no-cookies]
             [--no-check-certificate] [--continue] [--retries N] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             [--timeout SECONDS]
             url_or_id
gdown: error: unrecognized arguments: --id

Verifying downloaded file:
ls: cannot access '1D_png.zip': No such file or directory
1D_png.zip: cannot open `1D_png.zip' (No such file or directory)


In [ ]:
# Let's check the downloaded files to see if they are actual ZIP files or text files in disguise
!file 1D_png.zip
!head -n 10 1D_png.zip

1D_png.zip: HTML document, Unicode text, UTF-8 text, with very long lines (34529)








<!DOCTYPE html>
<html


In [ ]:
# Download directly from CERN's Zenodo server
!wget https://zenodo.org/records/4153431/files/SimpleCube++.zip

--2026-10-11 22:05:57--  https://zenodo.org/records/4153431/files/SimpleCube++.zip
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 137.138.52.235, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2113441199 (2.0G) [application/octet-stream]
Saving to: ‘SimpleCube++.zip’

SimpleCube++.zip    100%[===================>]   1.97G  8.07MB/s    in 7m 42s  

2026-10-11 22:13:39 (4.37 MB/s) - ‘SimpleCube++.zip’ saved [2113441199/2113441199]



In [ ]:
# Create a directory and extract silently
!mkdir -p dataset
!unzip -q SimpleCube++.zip -d dataset/

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from glob import glob

# --- Algorithm Adaptations (Returning B, G, R vectors) ---

def est_buchsbaum(img):
    # Ignore blacked-out cube area (pixels where all channels are near 0)
    valid_mask = np.any(img > 5, axis=-1)
    valid_pixels = img[valid_mask]
    if len(valid_pixels) == 0: return None
    return np.mean(valid_pixels, axis=0) # returns [B, G, R]

def est_blake(img_float, edge_thresh=20.0):
    grad_x = cv2.Sobel(img_float, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(img_float, cv2.CV_32F, 0, 1, ksize=3)
    mag_sq = grad_x**2 + grad_y**2
    edge_mask = mag_sq > (edge_thresh ** 2)
    valid_grad = np.sqrt(mag_sq) * edge_mask
    return np.array([np.sum(valid_grad[:,:,0]), np.sum(valid_grad[:,:,1]), np.sum(valid_grad[:,:,2])])

def est_grey_edge(img_float):
    blurred = cv2.GaussianBlur(img_float, (0, 0), sigmaX=3, sigmaY=3)
    grad_x = cv2.Sobel(blurred, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(blurred, cv2.CV_32F, 0, 1, ksize=3)
    edges = np.sqrt(grad_x**2 + grad_y**2)
    return np.array([np.mean(edges[:,:,0]), np.mean(edges[:,:,1]), np.mean(edges[:,:,2])])

def est_lee(img_float):
    g_channel = img_float[:, :, 1]
    # Filter out black pixels before calculating the 98th percentile
    bright_thresh = np.percentile(g_channel[g_channel > 5], 98)

    # 16-bit images max out at 65535
    spec_mask = (g_channel > bright_thresh) & (img_float[:,:,0] < 65000) & \
                (img_float[:,:,1] < 65000) & (img_float[:,:,2] < 65000)

    if np.sum(spec_mask) < 10:
        return None
    return np.array([np.mean(img_float[:,:,0][spec_mask]),
                     np.mean(img_float[:,:,1][spec_mask]),
                     np.mean(img_float[:,:,2][spec_mask])])

def est_huo(img_float, threshold=0.1321):
    img_8bit = np.clip(img_float / 256.0, 0, 255).astype(np.uint8)
    yuv = cv2.cvtColor(img_8bit, cv2.COLOR_BGR2YUV).astype(np.float32)

    Y = yuv[:, :, 0] + 1e-6
    U = yuv[:, :, 1] - 128.0
    V = yuv[:, :, 2] - 128.0
    F_yuv = (np.abs(U) + np.abs(V)) / Y
    gray_mask = (F_yuv < threshold) & (Y > 20)

    if np.sum(gray_mask) < 10:
        return None

    return np.array([np.mean(img_float[:,:,0][gray_mask]),
                     np.mean(img_float[:,:,1][gray_mask]),
                     np.mean(img_float[:,:,2][gray_mask])])

# --- Evaluation Metrics ---

def calculate_angular_error(est_bgr, true_rgb):
    if est_bgr is None:
        return None
    est_rgb = np.array([est_bgr[2], est_bgr[1], est_bgr[0]])
    est_norm = est_rgb / (np.linalg.norm(est_rgb) + 1e-6)
    true_norm = true_rgb / (np.linalg.norm(true_rgb) + 1e-6)
    cosine_sim = np.clip(np.dot(est_norm, true_norm), -1.0, 1.0)
    return np.degrees(np.arccos(cosine_sim))

# --- Main Benchmarking Loop ---

def run_benchmark():
    gt_files = glob("dataset/**/gt.csv", recursive=True)
    if not gt_files:
        print("No gt.csv files found! Check dataset path.")
        return

    results = []
    LIMIT_IMAGES = 2500
    processed_count = 0

    for gt_path in gt_files:
        base_dir = os.path.dirname(gt_path)
        png_dir = os.path.join(base_dir, "PNG")

        df_gt = pd.read_csv(gt_path)
        print(f"Reading {gt_path}. Available columns: {list(df_gt.columns)}")

        # Αντιστοίχιση των κατάλληλων στηλών για το ground truth RGB
        col_map = {}
        for col in ['r', 'g', 'b']:
            for candidate in [col, col.upper(), f'rgb_{col}', f'{col}_mean', f'mean_{col}']:
                if candidate in df_gt.columns:
                    col_map[col] = candidate
                    break

        if len(col_map) < 3:
            print(f"Error: Could not identify RGB ground truth columns in {df_gt.columns}")
            continue

        for index, row in df_gt.iterrows():
            if processed_count >= LIMIT_IMAGES:
                break

            img_name = str(row['image'])
            img_file = f"{img_name}.png" if not img_name.endswith('.png') else img_name
            path = os.path.join(png_dir, img_file)

            if not os.path.exists(path):
                continue

            true_rgb = np.array([row[col_map['r']], row[col_map['g']], row[col_map['b']]])

            img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
            if img is None:
                continue

            processed_count += 1
            if processed_count % 100 == 0:
                print(f"Processed {processed_count} images...")

            img_float = img.astype(np.float32)
            small_img = cv2.resize(img_float, (320, 240))

            est_gw = est_buchsbaum(small_img)
            est_bl = est_blake(small_img)
            est_ge = est_grey_edge(small_img)
            est_le = est_lee(small_img)
            est_hu = est_huo(small_img)

            row_data = {
                "Filename": img_file,
                "True_R/B": true_rgb[0] / (true_rgb[2] + 1e-6),
                "Buchsbaum_Error": calculate_angular_error(est_gw, true_rgb),
                "Blake_Error": calculate_angular_error(est_bl, true_rgb),
                "GreyEdge_Error": calculate_angular_error(est_ge, true_rgb),
                "Lee_Error": calculate_angular_error(est_le, true_rgb),
                "Huo_Error": calculate_angular_error(est_hu, true_rgb)
            }
            results.append(row_data)

    if not results:
        print("No images matched or processed successfully.")
        return

    df = pd.DataFrame(results)
    csv_path = "cube_plus_plus_benchmark.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nBenchmarking complete! Data saved to {csv_path}")
    print("\n--- MEDIAN ANGULAR ERRORS (Lower is better) ---")
    print(df.median(numeric_only=True).to_string())

if __name__ == "__main__":
    run_benchmark()

Reading dataset/SimpleCube++/train/gt.csv. Available columns: ['image', 'mean_r', 'mean_g', 'mean_b']
Processed 100 images...
Processed 200 images...
Processed 300 images...
Processed 400 images...
Processed 500 images...
Processed 600 images...
Processed 700 images...
Processed 800 images...
Processed 900 images...
Processed 1000 images...
Processed 1100 images...
Processed 1200 images...
Processed 1300 images...
Processed 1400 images...
Processed 1500 images...
Processed 1600 images...
Processed 1700 images...
Reading dataset/SimpleCube++/test/gt.csv. Available columns: ['image', 'mean_r', 'mean_g', 'mean_b']
Processed 1800 images...
Processed 1900 images...
Processed 2000 images...
Processed 2100 images...
Processed 2200 images...

Benchmarking complete! Data saved to cube_plus_plus_benchmark.csv

--- MEDIAN ANGULAR ERRORS (Lower is better) ---
True_R/B            0.620302
Buchsbaum_Error    12.823479
Blake_Error         3.322942
GreyEdge_Error      4.355256
Lee_Error           8.28

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from glob import glob
from scipy import stats

# --- Algorithm Adaptations (Returning Estimated R/B Score) ---

def score_buchsbaum(img):
    valid_mask = np.any(img > 5, axis=-1)
    valid_pixels = img[valid_mask]
    if len(valid_pixels) == 0:
        return np.nan
    b_avg, _, r_avg = np.mean(valid_pixels, axis=0)
    return float(r_avg / (b_avg + 1e-6))

def score_blake(img_float, edge_thresh=20.0):
    grad_x = cv2.Sobel(img_float, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(img_float, cv2.CV_32F, 0, 1, ksize=3)
    mag_sq = grad_x**2 + grad_y**2
    edge_mask = mag_sq > (edge_thresh ** 2)
    valid_grad = np.sqrt(mag_sq) * edge_mask

    b_edge = np.sum(valid_grad[:, :, 0])
    r_edge = np.sum(valid_grad[:, :, 2])
    return float(r_edge / (b_edge + 1e-6))

def score_grey_edge(img_float):
    blurred = cv2.GaussianBlur(img_float, (0, 0), sigmaX=3, sigmaY=3)
    grad_x = cv2.Sobel(blurred, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(blurred, cv2.CV_32F, 0, 1, ksize=3)
    edges = np.sqrt(grad_x**2 + grad_y**2)

    b_e = np.mean(edges[:, :, 0])
    r_e = np.mean(edges[:, :, 2])
    return float(r_e / (b_e + 1e-6))

def score_lee(img_float):
    g_channel = img_float[:, :, 1]
    non_black = g_channel[g_channel > 5]
    if len(non_black) == 0:
        return np.nan
    bright_thresh = np.percentile(non_black, 98)

    spec_mask = (g_channel > bright_thresh) & (img_float[:, :, 0] < 65000) & \
                (img_float[:, :, 1] < 65000) & (img_float[:, :, 2] < 65000)

    if np.sum(spec_mask) < 10:
        return np.nan
    b_spec = np.mean(img_float[:, :, 0][spec_mask])
    r_spec = np.mean(img_float[:, :, 2][spec_mask])
    return float(r_spec / (b_spec + 1e-6))

def score_huo(img_float, threshold=0.1321):
    img_8bit = np.clip(img_float / 256.0, 0, 255).astype(np.uint8)
    yuv = cv2.cvtColor(img_8bit, cv2.COLOR_BGR2YUV).astype(np.float32)

    Y = yuv[:, :, 0] + 1e-6
    U = yuv[:, :, 1] - 128.0
    V = yuv[:, :, 2] - 128.0
    F_yuv = (np.abs(U) + np.abs(V)) / Y
    gray_mask = (F_yuv < threshold) & (Y > 20)

    if np.sum(gray_mask) < 10:
        return np.nan
    b_pts = np.mean(img_float[:, :, 0][gray_mask])
    r_pts = np.mean(img_float[:, :, 2][gray_mask])
    return float(r_pts / (b_pts + 1e-6))

# --- Regression & Evaluation Pipeline ---

def run_regression_benchmark(limit_images=150):
    gt_files = glob("dataset/**/gt.csv", recursive=True)
    records = []
    processed = 0

    print("Extracting raw scores across dataset...")
    for gt_path in gt_files:
        base_dir = os.path.dirname(gt_path)
        png_dir = os.path.join(base_dir, "PNG")

        df_gt = pd.read_csv(gt_path, skipinitialspace=True)
        df_gt.columns = df_gt.columns.str.strip().str.lower()
        if len(df_gt.columns) == 1:
            df_gt = pd.read_csv(gt_path, sep=';', skipinitialspace=True)
            df_gt.columns = df_gt.columns.str.strip().str.lower()

        for _, row in df_gt.iterrows():
            if processed >= limit_images:
                break

            img_name = str(row.get('image', row.iloc[0])).strip()
            img_file = f"{img_name}.png" if not img_name.endswith('.png') else img_name
            path = os.path.join(png_dir, img_file)

            if not os.path.exists(path):
                continue

            r_val = float(row.get('r', row.get('mean_r')))
            b_val = float(row.get('b', row.get('mean_b')))
            true_rb = r_val / (b_val + 1e-6)

            img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
            if img is None:
                continue

            processed += 1
            img_float = img.astype(np.float32)
            small_img = cv2.resize(img_float, (320, 240))

            records.append({
                "Filename": img_file,
                "True_RB": true_rb,
                "Buchsbaum_Score": score_buchsbaum(small_img),
                "Blake_Score": score_blake(small_img),
                "GreyEdge_Score": score_grey_edge(small_img),
                "Lee_Score": score_lee(small_img),
                "Huo_Score": score_huo(small_img)
            })

    df = pd.DataFrame(records)
    df.to_csv("regression_raw_scores.csv", index=False)
    print(f"Collected raw warmth scores for {len(df)} images.\n")

    # --- Fitting Regressions ---
    methods = [
        ("Buchsbaum (Gray World)", "Buchsbaum_Score"),
        ("Blake (Retinex Edges)", "Blake_Score"),
        ("van de Weijer (Grey-Edge)", "GreyEdge_Score"),
        ("Lee (Specular Highlights)", "Lee_Score"),
        ("Huo (Gray Points)", "Huo_Score")
    ]

    regression_results = []

    for name, col in methods:
        valid_data = df[["True_RB", col]].dropna()
        x = valid_data[col].values      # Algorithm's predicted score
        y = valid_data["True_RB"].values # Ground truth warmth

        if len(x) < 5:
            continue

        # Linear regression: True_RB = slope * Method_Score + intercept
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
        r_squared = r_value ** 2

        # Root Mean Squared Error (RMSE) on regression-calibrated predictions
        predictions = slope * x + intercept
        rmse = np.sqrt(np.mean((predictions - y) ** 2))

        regression_results.append({
            "Method": name,
            "R² Score": r_squared,
            "Pearson (r)": r_value,
            "RMSE": rmse,
            "Slope (a)": slope,
            "Intercept (b)": intercept,
            "Valid Frames": len(x)
        })

    summary_df = pd.DataFrame(regression_results).sort_values(by="R² Score", ascending=False)

    print("=" * 80)
    print("                      REGRESSION BENCHMARK LEADERBOARD                          ")
    print("=" * 80)
    print(summary_df.to_string(index=False, justify="center"))
    print("=" * 80)

    best_method = summary_df.iloc[0]
    print(f"\nWinning Predictor: {best_method['Method']}")
    print(f"Correlation: r = {best_method['Pearson (r)']:.4f} | R² = {best_method['R² Score']:.4f}")
    print(f"Calibration Formula: Ground_Truth_RB = {best_method['Slope (a)']:.3f} * (Score) + ({best_method['Intercept (b)']:.3f})")

if __name__ == "__main__":
    run_regression_benchmark(limit_images=2500)

Extracting raw scores across dataset...
Collected raw warmth scores for 2234 images.

                      REGRESSION BENCHMARK LEADERBOARD                          
          Method           R² Score  Pearson (r)   RMSE    Slope (a)  Intercept (b)  Valid Frames
    Blake (Retinex Edges)  0.890677   0.943757   0.395090  1.173384    -0.215403         2234    
van de Weijer (Grey-Edge)  0.836833   0.914786   0.482676  1.375543    -0.381914         2234    
Lee (Specular Highlights)  0.808409   0.899116   0.538433  2.822673    -1.632991         2086    
   Buchsbaum (Gray World)  0.766020   0.875225   0.578002  5.429310    -4.243297         2234    
        Huo (Gray Points)  0.149516   0.386673   0.634053  3.220598    -2.257207          190    

Winning Predictor: Blake (Retinex Edges)
Correlation: r = 0.9438 | R² = 0.8907
Calibration Formula: Ground_Truth_RB = 1.173 * (Score) + (-0.215)
